# Install and Import Package

In [6]:
%pip install requests


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
import numpy as np
import pandas as pd

# --- Fix SSL certificate issue temporarily ---
import ssl
import nltk
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()
import time


[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


# Part 2 Step 1: Crawl a Real-World Dataset

In [8]:
 #Extract Reddit Data Variable
subreddit = 'stocks'
limit = 300
timeframe = 'year' #hour, day, week, month, year, all
listing = 'top' # controversial, best, hot, new, random, rising, top
 
def get_reddit(subreddit, listing, limit, timeframe):
    try:
        base_url = f'https://www.reddit.com/r/{subreddit}/{listing}.json?limit={limit}&t={timeframe}'
        request = requests.get(base_url, headers = {'User-agent': 'SDPA'})
    except:
        print('An Error Occured')
    return request
 
r = get_reddit(subreddit,listing,limit,timeframe)

In [9]:
import json
data = json.loads(r.content)
data

{'kind': 'Listing',
 'data': {'after': 't3_1knyf2u',
  'dist': 100,
  'modhash': '',
  'geo_filter': '',
  'children': [{'kind': 't3',
    'data': {'approved_at_utc': None,
     'subreddit': 'stocks',
     'selftext': "Trump Asks Supreme Court to Let Him Fire Top Agency Officials\n\nSummary by Bloomberg Al\n\n■ President Donald Trump has asked the US Supreme Court to allow him to immediately fire top officials at two independent agencies.\n\n■ The case is testing a 90-year-old Supreme Court ruling that lets Congress shield high-ranking officials from being fired by the president.\n\nThe outcome could determine whether Trump has the power to fire Federal Reserve Chair Jerome Powell and could also impact the job security of other agency officials.\n\n[https://www.bloomberg.com/news/articles/2025-04-09/trump-asks-supreme-court-to-let-him-fire-top-agency-officials](https://www.bloomberg.com/news/articles/2025-04-09/trump-asks-supreme-court-to-let-him-fire-top-agency-officials)\n\n**If this

In [10]:
temp = []
for i in range(100):
    post = data['data']['children'][i]['data']
    temp.append([post['title'], post['ups'], post['link_flair_text'], post['num_comments'], post['top_awarded_type'], post['selftext'], post['created_utc'], post['id']])
    
title = np.array(temp)
first_hundred = pd.DataFrame(title, columns=['Title', 'Upvotes', 'Flair', 'Num_Comments', 'Awards','Body', 'Created_UTC', 'Post_ID'])
print(first_hundred)

                                                Title Upvotes  \
0   PRESIDENT TRUMP JUST ASKED THE SUPREME COURT F...   52086   
1                                     Suicide hotline   49725   
2   Now we know. It was Retail CEOS who got to Tru...   47594   
3   America is going to get rocked. China, Japan, ...   46915   
4   China Officially Makes Statement Stating That ...   46880   
..                                                ...     ...   
95                  China is playing the US right now    5132   
96  Billionaire Ray Dalio: ‘I’m worried about some...    5121   
97  Is it possible that Trump’s tariffs are a mass...    5096   
98   Trump Taps Palantir to Compile Data on Americans    5064   
99  Donald Trump says US will set new tariff rates...    5055   

                       Flair Num_Comments Awards  \
0   misleading title / false         3617   None   
1                     Advice         1420   None   
2          Broad market news         2885   None   
3          

In [11]:
#Token for after parameter
after_token = data['data']['children'][99]['kind']+ '_' + data['data']['children'][99]['data']['id']
#After Function to get next 100 posts
def get_reddit_after(subreddit, listing, limit, timeframe, after_token):
    '''This function gets Reddit posts after a specific token'''
    try:
        base_url = f'https://www.reddit.com/r/{subreddit}/{listing}.json?limit={limit}&t={timeframe}&after={after_token}'
        request = requests.get(base_url, headers = {'User-agent': 'SDPA'})
    except:
        print('An Error Occured')
        return None
    return request
r_new = get_reddit_after(subreddit,listing,limit,timeframe, after_token)
new_data = json.loads(r_new.content)
new_data

temp = []
for i in range(100):
    post = new_data['data']['children'][i]['data']
    temp.append([post['title'], post['ups'], post['link_flair_text'], post['num_comments'], post['top_awarded_type'], post['selftext'],post['created_utc'], post['id']]), 
    
title = np.array(temp)
second_hundred = pd.DataFrame(title, columns=['Title', 'Upvotes', 'Flair', 'Num_Comments', 'Awards', 'Body', 'Created_UTC', 'Post_ID'])
print(second_hundred)

                                                Title Upvotes  \
0   This is a disaster of epic proportion”  Trump ...    4929   
1   Elon Musk defends $1 trillion pay package: ‘I ...    4836   
2   The era of American stock market exceptionalis...    4826   
3   Bloomberg: Markets are Discovering that the Re...    4727   
4       Tesla just got even more bad news from Europe    4684   
..                                                ...     ...   
95               Apple Announces $100B Share Buyback.    2891   
96    The Wall Street Journal explains Trump tariffs.    2871   
97  The Mar-A-Lago Accord - they are crashing the ...    2867   
98  Trump delays tariffs! For automakers. The rest...    2838   
99    Don’t Look at Stock Markets. Look at the Ports.    2829   

                            Flair Num_Comments Awards  \
0              Company Discussion          710   None   
1                    Company News          600   None   
2                            None          587  

In [12]:
df = pd.concat([first_hundred, second_hundred], ignore_index=True)
df.to_csv('reddit_python_top_300_posts_2023.csv', index=False)
df

,Title,Upvotes,Flair,Num_Comments,Awards,Body,Created_UTC,Post_ID
0,PRESIDENT TRUMP JUST ASKED THE SUPREME COURT F...,52086,misleading title / false,3617,None,Trump Asks Supreme Court to Let Him Fire Top A...,1744296575.0,1jvzroz
1,Suicide hotline,49725,Advice,1420,None,"The U.S. Suicide Hotline:\n\nDial 988, text 98...",1743800556.0,1jrmmdq
2,Now we know. It was Retail CEOS who got to Tru...,47594,Broad market news,2885,None,"As reported by Axios, Trump was shaken Monday ...",1745492967.0,1k6pihz
3,"America is going to get rocked. China, Japan, ...",46915,Broad market news,3595,None,[https://www.reuters.com/world/china-japan-sou...,1743438131.0,1jo76s8
4,China Officially Makes Statement Stating That ...,46880,Broad market news,3167,None,"China vows to stand firm, urges nations to res...",1745894404.0,1kadl0p
...,...,...,...,...,...,...,...,...
195,Apple Announces $100B Share Buyback.,2891,None,447,None,Apple’s CEO. “We were happy to welcome iPhone ...,1746132886.0,1kcjdo4
196,The Wall Street Journal explains Trump tariffs.,2871,None,457,None,"I had a similar analyses, but this coming from...",1745167872.0,1k3qwu1
197,The Mar-A-Lago Accord - they are crashing the ...,2867,Off topic: Political Bullshit,146,None,So I've posted around this paper by Stephen Mi...,1742045423.0,1jbv5dh
198,Trump delays tariffs! For automakers. The rest...,2838,None,339,None,[https://www.cnbc.com/2025/03/05/trump-grants-...,1741202648.0,1j4bg0p


# Frequent word in subreddit

In [13]:
# --- Fix SSL certificate issue temporarily ---
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# --- Now download stopwords safely ---
nltk.download('stopwords')

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


False

In [14]:
# ----- Download stopwords once -----
# (Run this only the first time; NLTK will remember afterward)
nltk.download('stopwords')

# ----- Define stopwords -----
stop_words = set(stopwords.words('english'))

# You can add your own extra words if you like
custom_stopwords = {'python', 'https', 'reddit', 'www', 'com'}
stop_words.update(custom_stopwords)

# ----- Prepare and clean titles -----
# Convert all titles to lowercase
title_clean = df['Title'].astype(str).str.lower()

# Combine all titles into one long text string
all_titles = ' '.join(title_clean)

# Extract only words (remove punctuation/numbers)
words = re.findall(r'\b[a-zA-Z]{3,}\b', all_titles)

# Filter out stopwords
filtered_words = [w for w in words if w not in stop_words]

# ----- Count the most frequent words -----
word_counts = Counter(filtered_words)

# Get top 30 most common words
top_words = word_counts.most_common(30)

# ----- Display results -----
print("\nTop 30 most frequent words in Reddit post titles:")
for word, count in top_words:
    print(f"{word:15} {count}")


Top 30 most frequent words in Reddit post titles:
trump           83
tariffs         47
says            27
china           26
tesla           25
market          23
tariff          16
musk            16
billion         12
stock           12
elon            10
powell          9
chinese         9
trade           9
canada          8
tsla            7
buy             6
breaking        6
since           6
back            6
walmart         6
prices          6
goods           6
american        5
white           5
house           5
amazon          5
economy         5
year            5
jobs            5


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


In [15]:
# ----- Import libraries -----
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords

# ----- Download stopwords once -----
# (Run this only the first time; NLTK will remember afterward)
nltk.download('stopwords')

# ----- Define stopwords -----
stop_words = set(stopwords.words('english'))

# You can add your own extra words if you like
custom_stopwords = {'python', 'https', 'reddit', 'www', 'com'}
stop_words.update(custom_stopwords)

# ----- Prepare and clean titles -----
# Convert all titles to lowercase
title_clean = df['Title'].astype(str).str.lower()

# Combine all titles into one long text string
all_titles = ' '.join(title_clean)

# Extract only words (remove punctuation/numbers)
words = re.findall(r'\b[a-zA-Z]{3,}\b', all_titles)

# Filter out stopwords
filtered_words = [w for w in words if w not in stop_words]

# ----- Count the most frequent words -----
word_counts = Counter(filtered_words)

# Get top 30 most common words
top_words = word_counts.most_common(30)

# ----- Display results -----
print("\nTop 30 most frequent words in Reddit post titles:")
for word, count in top_words:
    print(f"{word:15} {count}")


Top 30 most frequent words in Reddit post titles:
trump           83
tariffs         47
says            27
china           26
tesla           25
market          23
tariff          16
musk            16
billion         12
stock           12
elon            10
powell          9
chinese         9
trade           9
canada          8
tsla            7
buy             6
breaking        6
since           6
back            6
walmart         6
prices          6
goods           6
american        5
white           5
house           5
amazon          5
economy         5
year            5
jobs            5


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


# Sentiment Analyze Comment

In [20]:
results = []
for body in df['Body']:
    sentiment = sia.polarity_scores(body)
    sentiment['body'] = body
    compound = sentiment['compound']
    if compound >= 0.05:
        sentiment['Sentiment_Label'] = 'Positive'
    elif compound <= -0.05:
        sentiment['Sentiment_Label'] = 'Negative'
    else:
        sentiment['Sentiment_Label'] = 'Neutral'

    results.append(sentiment)
        
df_results = pd.DataFrame(results)
df_results


,neg,neu,pos,compound,body,Sentiment_Label
0,0.127,0.733,0.140,0.4094,Trump Asks Supreme Court to Let Him Fire Top A...,Positive
1,0.257,0.624,0.119,-0.8979,"The U.S. Suicide Hotline:\n\nDial 988, text 98...",Negative
2,0.048,0.864,0.088,0.5346,"As reported by Axios, Trump was shaken Monday ...",Positive
3,0.015,0.970,0.015,0.0258,[https://www.reuters.com/world/china-japan-sou...,Neutral
4,0.099,0.814,0.086,-0.5181,"China vows to stand firm, urges nations to res...",Negative
...,...,...,...,...,...,...
195,0.020,0.758,0.222,0.9601,Apple’s CEO. “We were happy to welcome iPhone ...,Positive
196,0.100,0.780,0.120,0.8770,"I had a similar analyses, but this coming from...",Positive
197,0.083,0.841,0.076,-0.6231,So I've posted around this paper by Stephen Mi...,Negative
198,0.182,0.723,0.095,-0.6763,[https://www.cnbc.com/2025/03/05/trump-grants-...,Negative


In [17]:
results = []
for headline in df['Title']:
    sentiment = sia.polarity_scores(headline)
    sentiment['headline'] = headline
    results.append(sentiment)
df_results = pd.DataFrame(results)
df_results


,neg,neu,pos,compound,headline
0,0.113,0.657,0.230,0.3612,PRESIDENT TRUMP JUST ASKED THE SUPREME COURT F...
1,0.818,0.182,0.000,-0.6705,Suicide hotline
2,0.000,1.000,0.000,0.0000,Now we know. It was Retail CEOS who got to Tru...
3,0.000,1.000,0.000,0.0000,"America is going to get rocked. China, Japan, ..."
4,0.000,0.772,0.228,0.6808,China Officially Makes Statement Stating That ...
...,...,...,...,...,...
195,0.000,0.645,0.355,0.2960,Apple Announces $100B Share Buyback.
196,0.000,1.000,0.000,0.0000,The Wall Street Journal explains Trump tariffs.
197,0.000,1.000,0.000,0.0000,The Mar-A-Lago Accord - they are crashing the ...
198,0.000,1.000,0.000,0.0000,Trump delays tariffs! For automakers. The rest...
